# NumPy: Arrays and Vectorized Math

The slides introduced NumPy as the foundation of the data-science stack: a third-party package that gives Python a new data structure, the n-dimensional array (`ndarray`). This notebook is the hands-on follow-up. We'll import NumPy, see why its arrays are faster than Python lists for numerical work, and practice the core operations: creating arrays, inspecting them, indexing and slicing, and reducing them with aggregations.

Pandas, which we'll meet Thursday, is built on top of NumPy. Everything you learn here carries forward.

## Import

By convention, NumPy is imported as `np`. You'll see this alias in every tutorial, textbook, and Stack Overflow answer; stick with it.

In [ ]:
import numpy as np

## Why NumPy?

NumPy is fast. How fast? Let's measure.

We'll create a list of one million integers and double every value, first with a list comprehension and then with a NumPy array. The `%timeit` magic runs the expression many times and reports an average.

In [ ]:
my_list = list(range(1_000_000))

Let's double it first with a traditional loop.

In [ ]:
%%timeit
double_loop = []
for val in my_list:
    double_loop.append(val * 2)

What about with a list comprehension?

In [ ]:
%timeit double_comp = [x * 2 for x in my_list]

On my machine `double_loop` ran in 15.9 ms per loop, and `double_comp` in 12.5 ms. Your numbers will differ.

In [ ]:
(15.9 - 12.5) / 15.9

About 21% faster.

Now the NumPy version...

First, convert `my_list` into a NumPy n-dimensional array object called `ndarray`.

In [ ]:
my_array = np.array(my_list)

In [ ]:
%timeit doubled = my_array * 2

1 ms = 1000 μs so this is...

In [ ]:
12.5 * 1000 / 488

About 25x faster than the best base Python has to offer.

Less trivial NumPy code is often **40-50 times faster**. That is not because NumPy is "optimized Python." It is because a NumPy array is a fundamentally different data structure:

- All elements share a single type (homogeneous), so the code that uses them can be streamlined.
- Elements are packed together in memory, so the CPU can stream through them efficiently.
- The arithmetic is written in C (a low-level language that can be optimized for the hardware) and applied to the whole array in one shot - no per-element Python loop.

The takeaway: **arrays are a different data structure, not a faster list.** Reach for them when you need to do math on many numbers at once.

## Creating Arrays

### From a list

Pass a list to `np.array` and you get a 1D array back.

In [ ]:
row = [1, 2, 3, 4, 5]
arr_1d = np.array(row)
print(arr_1d)
print(type(arr_1d))

### From nested lists

A list of lists becomes a 2D array - think of it as a table where each inner list is a row.

In [ ]:
table = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]
arr_2d = np.array(table)
print(arr_2d)

### Inspecting an array

Every array has three attributes worth knowing on day one:

- `dtype` - the type of the elements (e.g., `int64`, `float64`)
- `shape` - a tuple of axis lengths
- `ndim` - the number of axes (dimensions)

In [ ]:
print("dtype:", arr_2d.dtype)
print("shape:", arr_2d.shape)
print("ndim: ", arr_2d.ndim)

`shape` is always a tuple: `(3, 3)` means 3 rows by 3 columns. For a 1D array you'll see something like `(5,)` - the trailing comma is what makes it a one-element tuple.

In [ ]:
print("1D shape:", arr_1d.shape)
print("1D ndim: ", arr_1d.ndim)

### Creation helpers

You will often want an array of a given shape filled with a specific pattern. NumPy provides functions for the common cases.

In [ ]:
# all zeros - shape specified as a tuple
np.zeros((2, 3))

In [ ]:
# all ones
np.ones((2, 3))

In [ ]:
# like Python's range(), but returns an array
np.arange(10)

In [ ]:
# five evenly-spaced values from 0 to 1 (inclusive on both ends)
np.linspace(0, 1, 5)

`arange` and `linspace` are easy to mix up. `arange(start, stop, step)` matches `range` - you give it a step size. `linspace(start, stop, num)` gives you a specific number of evenly-spaced values.

## Indexing and Slicing

### 1D arrays behave like lists

Positions, slices, negative indices - all the same as lists.

In [ ]:
arr = np.array([10, 20, 30, 40, 50])

print("first:", arr[0])
print("last: ", arr[-1])
print("first three:", arr[:3])
print("every other:", arr[::2])

Note that NumPy arrays are represented differently than Python lists when printed, without commas.

### 2D arrays: row and column in one shot

With a 2D array you can use the familiar `arr[i][j]` pattern, but NumPy also accepts a single index with a comma: `arr[i, j]`. Prefer the comma form - it's shorter and scales to higher dimensions.

In [ ]:
arr_2d = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
])

print("middle element:", arr_2d[1, 1])
print("top-right:     ", arr_2d[0, 2])

### Whole rows and columns

A slice along an axis pulls a whole row or column. `:` by itself means "every position on this axis."

In [ ]:
# first row
print("first row:", arr_2d[0, :])

# first column
print("first col:", arr_2d[:, 0])

If you omit the column index, NumPy treats it as a full slice - so `arr_2d[0]` is the same as `arr_2d[0, :]`. Rows are easy. Columns require the explicit `[:, col]` form.

In [ ]:
# equivalent
print(arr_2d[0])
print(arr_2d[0, :])

Why is pulling a column so useful? Because in real data, rows are observations and columns are attributes. "Give me every value of `price`" is a column selection. In plain Python you'd write a loop:

In [ ]:
# the list version
col = []
for r in table:
    col.append(r[0])
print(col)

In [ ]:
# the NumPy version
print(arr_2d[:, 0])

### Worked examples

Pull the second column.

In [ ]:
arr_2d[:, 1]

Pull the last two rows.

In [ ]:
arr_2d[-2:, :]

The `-2:` means "from the second-to-last position to the end." Combined with `:` on the column axis, we get every column of those rows.

Pull the top-left 2x2 corner.

In [ ]:
arr_2d[:2, :2]

## Vectorized Aggregation

An aggregation reduces an array to a smaller summary - a sum, a mean, a maximum. NumPy provides these as methods on the array itself.

In [ ]:
arr_2d = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
])

print("sum: ", arr_2d.sum())
print("mean:", arr_2d.mean())
print("max: ", arr_2d.max())

By default these operate on every element, collapsing the whole array to a single number. That is rarely what you want with tabular data. Usually you want a sum or mean *per column* (one summary per attribute) or *per row* (one summary per observation).

That is what the `axis` parameter is for.

- `axis=0` collapses *along the rows* - so you get one value per column.
- `axis=1` collapses *along the columns* - so you get one value per row.

In [ ]:
# one value per column
print("column sums: ", arr_2d.sum(axis=0))
print("column means:", arr_2d.mean(axis=0))

In [ ]:
# one value per row
print("row sums: ", arr_2d.sum(axis=1))
print("row means:", arr_2d.mean(axis=1))

A quick way to remember: `axis=0` is the axis that "goes away." `arr_2d` has shape `(3, 3)`. `arr_2d.sum(axis=0)` has shape `(3,)` - the first axis (rows) is gone, leaving one value per column.

### Contrast with the loop

Computing column means with plain Python requires a nested loop and an accumulator:

In [ ]:
rows = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]

num_rows = len(rows)
num_cols = len(rows[0])
means = []
for j in range(num_cols):
    total = 0
    for i in range(num_rows):
        total += rows[i][j]
    means.append(total / num_rows)

print(means)

With NumPy it is one line:

In [ ]:
np.array(rows).mean(axis=0)

Once your data lives in an array, the operations you used to write as loops become single expressions - faster, shorter, and easier to read.

## What's Next

Thursday we'll meet Pandas: NumPy arrays with column names and mixed types, built for tabular data. Most of the operations you just learned carry over - indexing a column, aggregating down an axis - but with labels instead of integer positions.

Reminders:

- HW7 (install conda/Anaconda, ungraded) goes out today
- Project due Friday
- Final exam Monday 1:30-3:30

## Problems

**★ 1. Shape Report**

Write a function `shape_report(arr)` that takes a NumPy array and returns a tuple `(ndim, shape, dtype)`.

In [ ]:
# your code here

In [ ]:
result = shape_report(np.array([[1, 2], [3, 4], [5, 6]]))
assert result == (2, (3, 2), np.dtype("int64"))
print("Test passed!")

#### Solution

In [ ]:
def shape_report(arr):
    return (arr.ndim, arr.shape, arr.dtype)

print(shape_report(np.array([[1, 2], [3, 4], [5, 6]])))

**★ 2. Second Column**

Write a function `second_column(arr)` that returns the second column of a 2D NumPy array as a 1D array.

In [ ]:
# your code here

In [ ]:
arr = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
])
result = second_column(arr)
assert (result == np.array([2, 5, 8])).all()
print("Test passed!")

#### Solution

In [ ]:
def second_column(arr):
    return arr[:, 1]

arr = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(second_column(arr))

**★★ 3. Column Means**

Write a function `column_means(arr)` that returns the mean of each column in a 2D array. Use NumPy, not a loop.

In [ ]:
# your code here

In [ ]:
arr = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
])
result = column_means(arr)
assert (result == np.array([4.0, 5.0, 6.0])).all()
print("Test passed!")

#### Solution

In [ ]:
def column_means(arr):
    return arr.mean(axis=0)

arr = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]])
print(column_means(arr))

**★★ 4. Row Totals Above Threshold**

Write a function `rows_summing_above(arr, threshold)` that returns the row indices (as a list of ints) where the row sum exceeds `threshold`. Use `.sum(axis=1)` to get the row totals; a list comprehension is fine for collecting the indices.

In [ ]:
# your code here

In [ ]:
arr = np.array([
    [1, 2, 3],    # sum =  6
    [4, 5, 6],    # sum = 15
    [7, 8, 9],    # sum = 24
])
assert rows_summing_above(arr, 10) == [1, 2]
assert rows_summing_above(arr, 20) == [2]
assert rows_summing_above(arr, 100) == []
print("All tests passed!")

#### Solution

In [ ]:
def rows_summing_above(arr, threshold):
    totals = arr.sum(axis=1)
    return [i for i, t in enumerate(totals) if t > threshold]

arr = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(rows_summing_above(arr, 10))